# Generation OK retrieval probe

Run the full Python RAG pipeline on the 10 source-specific questions for the Generation OK issues and inspect what happens at each stage: retrieval chunks, aggregated sections, selector/context, final answer, and displayed sources.

Expected env vars:

- `SCW_POSTGRES_DSN_STAGING` for staging
- `SCW_POSTGRES_DSN_PROD` for production
- provider vars used by the pipeline: `ALBERT_API_KEY`, `ALBERT_BASE_URL`, `SCALEWAY_API_KEY`, `SCALEWAY_BASE_URL`


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import Markdown, display
from psycopg.rows import dict_row


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "packages/rag-pipeline").exists():
            return candidate
    raise RuntimeError("Could not find assistant-rh repository root from current working directory.")


REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env")

for package_src in (REPO_ROOT / "packages/rag-pipeline/src", REPO_ROOT / "packages/shared-config/src"):
    package_src_str = str(package_src)
    if package_src_str not in sys.path:
        sys.path.insert(0, package_src_str)

print(f"Repo root: {REPO_ROOT}")


## Run controls

Set `TARGET_ENV` to `staging` or `prod`. The notebook defaults to a one-question Service-Public smoke run so it does not look frozen while full LLM calls are running. For the full 30-question pass, set `QUESTION_SET = "all"` and `RUN_FIRST_N = None`.

In [ ]:
TARGET_ENV = "staging"  # "staging" or "prod"
QUESTION_SET = "service_public"  # "all", "dgafp_legifrance", "matte", "service_public"
LIMIT_PER_SET = 10
RUN_FIRST_N = 10  # Keep 1 for a smoke run; set to None to run every selected question.
TOP_N_CHUNKS = 12
TOP_N_SECTIONS = 10
RAISE_ON_ERROR = False
DB_CONNECT_TIMEOUT_S = 10
ALLOW_SCW_DSN_FALLBACK_FOR_STAGING = True

DSN_ENV_BY_TARGET = {
    "staging": "SCW_POSTGRES_DSN_STAGING",
    "prod": "SCW_POSTGRES_DSN_PROD",
}


def with_connect_timeout(dsn: str, seconds: int) -> str:
    if "connect_timeout=" in dsn:
        return dsn
    if "://" not in dsn:
        return f"{dsn} connect_timeout={seconds}"
    separator = "&" if "?" in dsn else "?"
    return f"{dsn}{separator}connect_timeout={seconds}"


def resolve_dsn(target_env: str) -> tuple[str, str]:
    key = DSN_ENV_BY_TARGET.get(target_env)
    if not key:
        raise ValueError(f"Unsupported TARGET_ENV={target_env!r}; expected one of {sorted(DSN_ENV_BY_TARGET)}")
    value = os.getenv(key, "").strip()
    source = key
    if not value and target_env == "staging" and ALLOW_SCW_DSN_FALLBACK_FOR_STAGING:
        value = os.getenv("SCW_POSTGRES_DSN", "").strip()
        source = "SCW_POSTGRES_DSN fallback for staging"
    if not value:
        raise RuntimeError(f"Missing {key}. Add it to .env or the notebook kernel environment.")
    return with_connect_timeout(value, DB_CONNECT_TIMEOUT_S), source


def validate_provider_env() -> None:
    required = ["ALBERT_API_KEY", "ALBERT_BASE_URL", "SCALEWAY_API_KEY", "SCALEWAY_BASE_URL"]
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Missing provider env vars for full pipeline run: {', '.join(missing)}")


SELECTED_DSN, SELECTED_DSN_SOURCE = resolve_dsn(TARGET_ENV)
# Some pipeline helpers still resolve the canonical env var internally
# for prompts/acronyms. Pin it to the selected notebook target so all stages
# hit the same database.
os.environ["APP_DB_TARGET"] = "scaleway"
os.environ["APP_SCALEWAY_ENV"] = TARGET_ENV
os.environ["SCW_POSTGRES_DSN"] = SELECTED_DSN
validate_provider_env()
print(f"Target: {TARGET_ENV} ({SELECTED_DSN_SOURCE}, connect_timeout={DB_CONNECT_TIMEOUT_S}s)")


## Question sets

- DGAFP/Légifrance comes from the committed conformance fixture for issue #150.
- Service-Public is embedded from issue #155.
- MATTE is loaded from the selected database because issue #153 does not currently contain its 10 questions.

In [ ]:
@dataclass(frozen=True)
class QuestionCase:
    id: str
    family: str
    expected_family: str
    question: str
    origin: str
    gold_sources: str = ""
    document_title: str = ""


SERVICE_PUBLIC_QUESTIONS = [
    "Quel est le montant de la prise en charge des frais de transports en commun domicile-travail pour un agent public ?",
    "Quels agents publics peuvent bénéficier du remboursement des frais de transport domicile-travail ?",
    "Quelles démarches un agent public doit-il effectuer pour demander le remboursement de son abonnement de transport ?",
    "Un agent public à temps partiel bénéficie-t-il du même remboursement de transport qu'un agent à temps complet ?",
    "Quelles sont les conditions pour bénéficier du supplément familial de traitement dans la fonction publique ?",
    "Comment est calculé le supplément familial de traitement selon le nombre d'enfants ?",
    "Quelles sont les règles applicables au télétravail dans la fonction publique ?",
    "Quelles sont les conditions de recours à la rupture conventionnelle pour un agent public ?",
    "Quelles démarches un agent public doit-il suivre pour demander un congé parental ?",
    "Quels sont les droits d'un agent public en cas de congé de maladie ordinaire ?",
]


def load_dgafp_legifrance_questions(limit: int = 10) -> list[QuestionCase]:
    path = REPO_ROOT / "tests/conformance/queries.legifrance-source-check.jsonl"
    cases: list[QuestionCase] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            obj = json.loads(line)
            tags = set(obj.get("tags") or [])
            if "adversarial" in tags:
                continue
            if "source-specific" not in tags:
                continue
            cases.append(
                QuestionCase(
                    id=str(obj.get("id") or f"legifrance-q{len(cases) + 1}"),
                    family="dgafp_legifrance",
                    expected_family="dgafp_legifrance",
                    question=str(obj["query"]),
                    origin="issue #150 / tests/conformance/queries.legifrance-source-check.jsonl",
                )
            )
            if len(cases) >= limit:
                break
    if len(cases) != limit:
        raise RuntimeError(f"Expected {limit} DGAFP/Légifrance questions, found {len(cases)}")
    return cases


def load_service_public_questions(limit: int = 10) -> list[QuestionCase]:
    cases = [
        QuestionCase(
            id=f"service-public-q{idx:02d}",
            family="service_public",
            expected_family="service_public",
            question=question,
            origin="issue #155",
        )
        for idx, question in enumerate(SERVICE_PUBLIC_QUESTIONS[:limit], start=1)
    ]
    if len(cases) != limit:
        raise RuntimeError(f"Expected {limit} Service-Public questions, found {len(cases)}")
    return cases


def load_matte_questions(dsn: str, limit: int = 10) -> list[QuestionCase]:
    diverse_sql = """
        WITH ranked AS (
            SELECT
                gq.id,
                gq.question,
                gq.gold_sources,
                d.title AS document_title,
                row_number() OVER (PARTITION BY gq.gold_sources ORDER BY gq.id) AS per_doc_rank
            FROM goldset_questions_v2 gq
            JOIN rag_documents d ON btrim(gq.gold_sources) = btrim(d.short_id)
            WHERE d.publisher = 'MATTE'
              AND gq.question IS NOT NULL
              AND btrim(gq.question) <> ''
              AND gq.gold_sources IS NOT NULL
              AND btrim(gq.gold_sources) <> ''
        )
        SELECT id, question, gold_sources, document_title
        FROM ranked
        WHERE per_doc_rank = 1
        ORDER BY id
        LIMIT %s
    """
    fallback_sql = """
        SELECT gq.id, gq.question, gq.gold_sources, d.title AS document_title
        FROM goldset_questions_v2 gq
        JOIN rag_documents d ON btrim(gq.gold_sources) = btrim(d.short_id)
        WHERE d.publisher = 'MATTE'
          AND gq.question IS NOT NULL
          AND btrim(gq.question) <> ''
          AND gq.gold_sources IS NOT NULL
          AND btrim(gq.gold_sources) <> ''
        ORDER BY gq.id
        LIMIT %s
    """
    with psycopg.connect(dsn, row_factory=dict_row) as conn:
        with conn.cursor() as cur:
            cur.execute(diverse_sql, (limit,))
            rows = [dict(row) for row in cur.fetchall()]
            if len(rows) < limit:
                cur.execute(fallback_sql, (limit,))
                rows = [dict(row) for row in cur.fetchall()]
    if len(rows) < limit:
        raise RuntimeError(f"Expected at least {limit} MATTE goldset questions in {TARGET_ENV}, found {len(rows)}")
    return [
        QuestionCase(
            id=f"matte-goldset-{row['id']}",
            family="matte",
            expected_family="matte",
            question=str(row["question"]),
            origin=f"{TARGET_ENV} goldset_questions_v2 MATTE fallback",
            gold_sources=str(row.get("gold_sources") or ""),
            document_title=str(row.get("document_title") or ""),
        )
        for row in rows[:limit]
    ]


def load_question_bank(dsn: str, question_set: str, limit_per_set: int = 10) -> list[QuestionCase]:
    cases: list[QuestionCase] = []
    if question_set in {"all", "dgafp_legifrance"}:
        cases.extend(load_dgafp_legifrance_questions(limit_per_set))
    if question_set in {"all", "matte"}:
        cases.extend(load_matte_questions(dsn, limit_per_set))
    if question_set in {"all", "service_public"}:
        cases.extend(load_service_public_questions(limit_per_set))
    return cases


ALL_CASES = load_question_bank(SELECTED_DSN, QUESTION_SET, LIMIT_PER_SET)
if not ALL_CASES:
    raise RuntimeError(f"No questions selected for QUESTION_SET={QUESTION_SET!r}")
CASES_TO_RUN = ALL_CASES[:RUN_FIRST_N] if RUN_FIRST_N is not None else ALL_CASES

QUESTIONS_DF = pd.DataFrame([case.__dict__ for case in ALL_CASES])
RUN_QUEUE_DF = pd.DataFrame([case.__dict__ for case in CASES_TO_RUN])
print(f"Loaded {len(ALL_CASES)} question(s); queued {len(CASES_TO_RUN)} for execution.")
display(Markdown("**All selected questions**"))
display(QUESTIONS_DF)
display(Markdown("**Execution queue**"))
display(RUN_QUEUE_DF)


## Pipeline setup

In [ ]:
from assistant_rh_rag_pipeline import create_pipeline, get_default_config
from assistant_rh_rag_pipeline.admin import get_rag_config
from assistant_rh_rag_pipeline.config import ContextMode, SearchMode


def build_streamlit_config(runtime_config: Any):
    """Mirror apps/streamlit-ui/pages/01_Chatbot.py RAG V3 config mapping."""
    config = get_default_config()
    mode_map = {"standard": ContextMode.STANDARD, "wide": ContextMode.WIDE}
    config.context.context_mode = mode_map.get(getattr(runtime_config, "v3_context_mode", "standard"), ContextMode.STANDARD)
    config.context.token_budget = getattr(runtime_config, "v3_token_budget", 8000)
    config.context.doc_entire_threshold = getattr(runtime_config, "v3_doc_entire_threshold", 3500)
    config.selector.enabled = getattr(runtime_config, "v3_enable_selector", True)
    config.selector.model = getattr(runtime_config, "v3_selector_model", "openweight-large")
    config.selector.prompt_name = getattr(runtime_config, "v3_selector_prompt_name", "v3_selector_business.md")
    config.query_processor.enable_intent_gating = getattr(runtime_config, "enable_intent_gating", False)
    config.query_processor.enable_acronym_expansion = getattr(runtime_config, "enable_query_expansion", True)
    config.query_processor.intent_prompt_name = getattr(runtime_config, "v3_intent_prompt_name", "intent_unified.md")
    config.retrieval.tables = list(getattr(runtime_config, "v3_tables", None) or ["matte", "service_public", "dgafp", "rgrh"])
    config.retrieval.enable_chunks_test = getattr(runtime_config, "v3_enable_chunks_test", True)
    config.retrieval.initial_top_k = getattr(runtime_config, "v3_initial_top_k", 10)
    config.retrieval.alpha = getattr(runtime_config, "v3_alpha", 0.5)
    config.aggregation.enable_section_reranker = getattr(runtime_config, "v3_enable_reranker", True)
    config.aggregation.section_rerank_top_k = getattr(runtime_config, "v3_rerank_top_k", 5)
    search_mode_map = {"semantic": SearchMode.SEMANTIC, "hybrid": SearchMode.HYBRID, "lexical": SearchMode.LEXICAL}
    config.retrieval.search_mode = search_mode_map.get(getattr(runtime_config, "v3_search_mode", "semantic"), SearchMode.SEMANTIC)
    config.generation.model = getattr(runtime_config, "v3_generator_model", "openweight-large")
    config.generation.temperature = getattr(runtime_config, "v3_temperature", 0.0)
    config.generation.system_prompt_name = getattr(runtime_config, "v3_system_prompt_name", "system_prompt_V6_optimized.md")
    config.verbose = getattr(runtime_config, "verbose_mode", False)
    return config


RUNTIME_RAG_CONFIG = get_rag_config()
PIPELINE_CONFIG = build_streamlit_config(RUNTIME_RAG_CONFIG)
PIPELINE = create_pipeline(PIPELINE_CONFIG, dsn=SELECTED_DSN)
{
    "target_env": TARGET_ENV,
    "runtime_config": RUNTIME_RAG_CONFIG.to_dict(),
    "pipeline_config": PIPELINE_CONFIG.to_dict(),
}


## Run full pipeline

In [ ]:
EXPECTED_TERMS = {
    "dgafp_legifrance": ("dgafp", "legifrance", "légifrance"),
    "matte": ("matte",),
    "service_public": ("service-public", "service_public", "service public"),
}


def compact_text(value: Any, limit: int = 360) -> str:
    text = " ".join(str(value or "").split())
    return text if len(text) <= limit else f"{text[: limit - 1]}…"


def matches_expected_family(row: dict[str, Any], expected_family: str) -> bool:
    terms = EXPECTED_TERMS[expected_family]
    haystack = " ".join(str(row.get(key, "") or "") for key in ("table", "source_table", "publisher", "document_title", "origin")).lower()
    return any(term in haystack for term in terms)


def attempts_from_result(result: Any) -> list[dict[str, Any]]:
    attempts = (result.metadata or {}).get("retrieval_attempts") or []
    return [attempt for attempt in attempts if isinstance(attempt, dict)]


def chunk_rows(result: Any, expected_family: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for attempt in attempts_from_result(result):
        for rank, chunk in enumerate(attempt.get("retrieved_chunks") or [], start=1):
            if not isinstance(chunk, dict):
                continue
            row = {
                "attempt": attempt.get("name", ""),
                "rank": rank,
                "score": chunk.get("score"),
                "table": chunk.get("table") or chunk.get("source_table") or "",
                "chunk_id": chunk.get("chunk_id", ""),
                "section_id": chunk.get("section_id", ""),
                "retrieval_path": chunk.get("retrieval_path", ""),
                "heading_match_score": chunk.get("heading_match_score"),
                "preview": compact_text(chunk.get("preview"), 420),
            }
            row["expected_hit"] = matches_expected_family(row, expected_family)
            rows.append(row)
    return rows


def section_rows(result: Any, expected_family: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for attempt in attempts_from_result(result):
        for rank, section in enumerate(attempt.get("aggregated_sections") or [], start=1):
            if not isinstance(section, dict):
                continue
            row = {
                "attempt": attempt.get("name", ""),
                "rank": rank,
                "score": section.get("score"),
                "publisher": section.get("publisher", ""),
                "heading": section.get("heading", ""),
                "chunk_count": section.get("chunk_count"),
                "token_estimate": section.get("token_estimate"),
                "document_id": section.get("document_id", ""),
                "section_id": section.get("section_id", ""),
            }
            row["expected_hit"] = matches_expected_family(row, expected_family)
            rows.append(row)
    return rows


def context_rows(result: Any, expected_family: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for rank, item in enumerate(result.context_items or [], start=1):
        metadata = getattr(item, "metadata", {}) or {}
        row = {
            "rank": rank,
            "score": round(float(getattr(item, "score", 0.0) or 0.0), 4),
            "publisher": getattr(item, "publisher", "") or "",
            "heading": getattr(item, "heading", "") or "",
            "document_title": getattr(item, "document_title", "") or "",
            "tokens": getattr(item, "token_estimate", 0),
            "section_id": str(getattr(item, "section_id", "") or ""),
            "doc_id": str(metadata.get("doc_id", "") or ""),
            "is_doc_entire": bool(metadata.get("is_doc_entire", False)),
            "preview": compact_text(getattr(item, "content", ""), 520),
        }
        row["expected_hit"] = matches_expected_family(row, expected_family)
        rows.append(row)
    return rows


def source_rows(result: Any, expected_family: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for rank, source in enumerate(result.sources or [], start=1):
        row = {
            "rank": rank,
            "publisher": source.get("publisher", ""),
            "heading": source.get("heading", ""),
            "document_title": source.get("document_title", ""),
            "document_url": source.get("document_url", ""),
            "score": source.get("score"),
        }
        row["expected_hit"] = matches_expected_family(row, expected_family)
        rows.append(row)
    return rows


def summarize_case(case: QuestionCase, result: Any | None, error: str = "") -> dict[str, Any]:
    if result is None:
        return {**case.__dict__, "status": "error", "error": error}
    metadata = result.metadata or {}
    chunks = chunk_rows(result, case.expected_family)
    sections = section_rows(result, case.expected_family)
    contexts = context_rows(result, case.expected_family)
    sources = source_rows(result, case.expected_family)
    answer = result.answer or ""
    return {
        **case.__dict__,
        "status": "ok",
        "error": "",
        "needs_legal_search": metadata.get("needs_legal_search"),
        "tables_searched": ", ".join(metadata.get("tables_searched") or []),
        "selector_decision": metadata.get("selector_decision"),
        "selector_before": metadata.get("selector_items_before"),
        "selector_after": metadata.get("selector_items_after"),
        "selector_retry_triggered": metadata.get("selector_retry_triggered"),
        "selector_retry_succeeded": metadata.get("selector_retry_succeeded"),
        "chunks_total": len(chunks),
        "chunks_expected_hits": sum(1 for row in chunks if row["expected_hit"]),
        "sections_total": len(sections),
        "sections_expected_hits": sum(1 for row in sections if row["expected_hit"]),
        "context_total": len(contexts),
        "context_expected_hits": sum(1 for row in contexts if row["expected_hit"]),
        "sources_total": len(sources),
        "sources_expected_hits": sum(1 for row in sources if row["expected_hit"]),
        "answer_abstained": "Je n'ai pas trouvé d'informations suffisamment pertinentes" in answer,
        "answer_preview": compact_text(answer, 260),
        "pipeline_total_ms": round(sum(float(v or 0.0) for v in (result.timing or {}).values() if isinstance(v, (int, float))), 1),
    }


def run_case(case: QuestionCase) -> dict[str, Any]:
    started = time.perf_counter()
    try:
        result = PIPELINE.run_with_trace(case.question)
        elapsed_ms = (time.perf_counter() - started) * 1000
        return {
            "case": case,
            "result": result,
            "error": "",
            "summary": {**summarize_case(case, result), "wall_ms": round(elapsed_ms, 1)},
        }
    except Exception as exc:
        if RAISE_ON_ERROR:
            raise
        return {
            "case": case,
            "result": None,
            "error": repr(exc),
            "summary": summarize_case(case, None, repr(exc)),
        }


RUNS = []
for index, case in enumerate(CASES_TO_RUN, start=1):
    print(f"[{index}/{len(CASES_TO_RUN)}] start {case.id}: {case.question[:90]}", flush=True)
    run = run_case(case)
    RUNS.append(run)
    print(f"[{index}/{len(CASES_TO_RUN)}] done {case.id}: status={run['summary'].get('status')} error={bool(run['error'])}", flush=True)

SUMMARY_DF = pd.DataFrame([run["summary"] for run in RUNS])
SUMMARY_DF


## Summary by source family

In [ ]:
hit_columns = [
    "chunks_expected_hits",
    "sections_expected_hits",
    "context_expected_hits",
    "sources_expected_hits",
]

family_summary = (
    SUMMARY_DF.groupby("family", dropna=False)
    .agg(
        questions=("id", "count"),
        errors=("error", lambda values: sum(bool(value) for value in values)),
        retrieval_hits=("chunks_expected_hits", lambda values: sum(value > 0 for value in values)),
        aggregation_hits=("sections_expected_hits", lambda values: sum(value > 0 for value in values)),
        context_hits=("context_expected_hits", lambda values: sum(value > 0 for value in values)),
        displayed_source_hits=("sources_expected_hits", lambda values: sum(value > 0 for value in values)),
        abstentions=("answer_abstained", "sum"),
    )
    .reset_index()
)

display(family_summary)
display(SUMMARY_DF[["id", "family", "question", "needs_legal_search", "selector_decision", *hit_columns, "answer_abstained", "error"]])


## Detailed stage inspection

Rows with `expected_hit = True` match the expected source family for that question set.

In [ ]:
def display_table(rows: list[dict[str, Any]], columns: list[str], *, max_rows: int | None = None) -> None:
    if not rows:
        display(Markdown("_No rows._"))
        return
    df = pd.DataFrame(rows)
    if max_rows is not None:
        df = df.head(max_rows)
    existing_columns = [column for column in columns if column in df.columns]
    df = df[existing_columns]
    if "expected_hit" in df.columns:
        display(df.style.apply(lambda row: ["background-color: #e8f5e9" if row.get("expected_hit") else "" for _ in row], axis=1))
    else:
        display(df)


def display_run(run: dict[str, Any]) -> None:
    case: QuestionCase = run["case"]
    result = run["result"]
    summary = run["summary"]
    display(Markdown(f"### {case.id} · {case.family}"))
    display(Markdown(f"**Question**: {case.question}"))
    if case.gold_sources:
        display(Markdown(f"**Gold source fallback**: `{case.gold_sources}` · {case.document_title}"))
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
    if result is None:
        return

    display(Markdown("#### Retrieved chunks"))
    display_table(
        chunk_rows(result, case.expected_family),
        ["attempt", "rank", "expected_hit", "score", "table", "retrieval_path", "heading_match_score", "chunk_id", "section_id", "preview"],
        max_rows=TOP_N_CHUNKS,
    )

    display(Markdown("#### Aggregated sections"))
    display_table(
        section_rows(result, case.expected_family),
        ["attempt", "rank", "expected_hit", "score", "publisher", "heading", "chunk_count", "token_estimate", "section_id", "document_id"],
        max_rows=TOP_N_SECTIONS,
    )

    display(Markdown("#### Final context items"))
    display_table(
        context_rows(result, case.expected_family),
        ["rank", "expected_hit", "score", "publisher", "heading", "document_title", "tokens", "is_doc_entire", "section_id", "preview"],
        max_rows=None,
    )

    display(Markdown("#### Displayed sources"))
    display_table(
        source_rows(result, case.expected_family),
        ["rank", "expected_hit", "score", "publisher", "heading", "document_title", "document_url"],
        max_rows=None,
    )

    selector_reason = (result.metadata or {}).get("selector_reasoning") or (result.metadata or {}).get("selector_rejection_reason") or ""
    if selector_reason:
        display(Markdown("#### Selector reasoning"))
        display(Markdown(compact_text(selector_reason, 1200)))

    display(Markdown("#### Answer"))
    display(Markdown(result.answer or "_Empty answer._"))


for run in RUNS:
    display_run(run)


## Optional JSON export

In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    export_payload = {
        "created_at": datetime.now(tz=UTC).isoformat(),
        "target_env": TARGET_ENV,
        "question_set": QUESTION_SET,
        "config": PIPELINE_CONFIG.to_dict(),
        "summary": SUMMARY_DF.to_dict(orient="records"),
        "details": [
            {
                "case": run["case"].__dict__,
                "error": run["error"],
                "answer": getattr(run["result"], "answer", "") if run["result"] else "",
                "sources": getattr(run["result"], "sources", []) if run["result"] else [],
                "timing": getattr(run["result"], "timing", {}) if run["result"] else {},
                "metadata": getattr(run["result"], "metadata", {}) if run["result"] else {},
            }
            for run in RUNS
        ],
    }
    output_path = REPO_ROOT / "tmp" / f"generation_ok_retrieval_probe_{TARGET_ENV}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(export_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"Wrote {output_path}")
